# Universal positional k-mer bias in the mature-miRNA sequence repertoire

This notebook performs the **universal sequence-repertoire analysis**:

1. Read the miRBase `mature.fa` file.
2. Strictly validate RNA sequences against `A/C/G/U`.
3. Globally deduplicate exact mature sequences.
4. Count **overlapping** 2-mers and 3-mers at every zero-based start position.
5. Normalize each position by the number of unique sequences eligible to
   contribute a complete k-mer at that position.
6. Export counts, frequencies, eligibility denominators, tidy tables, audit
   information, and editable SVG heatmaps.

Global deduplication is intentional here because the estimand is the unique
mature-miRNA **sequence repertoire**. It must not be reused for taxon-specific
analyses, where deduplication should occur within each organism.


## Dependencies

The notebook requires Python 3.10 or later plus `pandas`, `numpy`,
`matplotlib`, and `seaborn`.

If needed, uncomment and run the following in a separate cell:

```python
# %pip install pandas numpy matplotlib seaborn
```


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import platform
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", mpl.__version__)
print("seaborn:", sns.__version__)


## Configuration


In [ ]:
# Input and output paths
FASTA_PATH = Path(r"C:\Users\GCV\Downloads\mature.fa")
OUTPUT_DIR = Path.cwd() / "universal_kmer_outputs"

# Strict file-version checks for the mature.fa file audited on 25 July 2026.
# Set either value to None if intentionally analysing another release.
EXPECTED_RAW_RECORDS = 48_885
EXPECTED_UNIQUE_SEQUENCES = 30_721
EXPECTED_SHA256 = (
    "3c521fc9bea3c7993e71cf188b11caf73e2d40ac83454e32876455317cc6d342"
)

# Scientific settings
VALID_BASES = frozenset("ACGU")
BASE_ORDER = ("A", "C", "G", "U")
K_VALUES = (2, 3)

# Plot only positions at which at least this fraction of the globally
# unique repertoire can contribute a complete k-mer. Full-position data
# are always retained in the CSV outputs.
MIN_ELIGIBLE_FRACTION_FOR_PLOT = 0.10

# Set to an integer to override the support rule, e.g. 22 to plot 0..22.
# Leave as None for the defensible denominator-based rule.
MAX_START_POSITION_OVERRIDE = None

# Figure design, following the supplied reference SVG.
COLOR_MAP = "RdPu"
FIGURE_BACKGROUND = "#fffafa"
GRID_COLOR = "#eadfdf"
FONT_FAMILY = "Arial"
SVG_TRANSPARENT = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", FASTA_PATH.resolve())
print("Outputs:", OUTPUT_DIR.resolve())


## FASTA parsing and strict validation


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_fasta(path: Path) -> list[dict[str, str]]:
    '''
    Parse a FASTA file without altering the sequence alphabet.

    Returning the complete header makes every exclusion or validation
    failure traceable to its source record.
    '''
    records: list[dict[str, str]] = []
    header: str | None = None
    sequence_parts: list[str] = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    records.append(
                        {
                            "header": header,
                            "sequence": "".join(sequence_parts).upper(),
                        }
                    )
                header = line[1:].strip()
                sequence_parts = []
            else:
                if header is None:
                    raise ValueError(
                        f"Sequence encountered before a FASTA header "
                        f"at line {line_number}."
                    )
                sequence_parts.append(line)

    if header is not None:
        records.append(
            {
                "header": header,
                "sequence": "".join(sequence_parts).upper(),
            }
        )

    if not records:
        raise ValueError(f"No FASTA records were found in {path}.")
    return records


def validate_rna_records(
    records: list[dict[str, str]],
    valid_bases: frozenset[str] = VALID_BASES,
) -> None:
    invalid_records = []
    empty_records = []

    for record in records:
        sequence = record["sequence"]
        if not sequence:
            empty_records.append(record["header"])
            continue
        invalid = sorted(set(sequence) - valid_bases)
        if invalid:
            invalid_records.append(
                {
                    "header": record["header"],
                    "invalid_characters": "".join(invalid),
                }
            )

    if empty_records or invalid_records:
        examples = invalid_records[:10]
        raise ValueError(
            "FASTA validation failed. "
            f"Empty records: {len(empty_records)}; "
            f"records with non-ACGU characters: {len(invalid_records)}. "
            f"First invalid examples: {examples}"
        )


In [ ]:
if not FASTA_PATH.is_file():
    raise FileNotFoundError(f"FASTA file not found: {FASTA_PATH}")

fasta_sha256 = sha256_file(FASTA_PATH)
records = parse_fasta(FASTA_PATH)
validate_rna_records(records)

accessions = [record["header"].split()[1] for record in records]
mirna_ids = [record["header"].split()[0] for record in records]

if len(set(accessions)) != len(accessions):
    raise AssertionError("Duplicate MIMAT accessions were detected.")
if len(set(mirna_ids)) != len(mirna_ids):
    raise AssertionError("Duplicate mature-miRNA identifiers were detected.")

if EXPECTED_RAW_RECORDS is not None:
    assert len(records) == EXPECTED_RAW_RECORDS, (
        f"Expected {EXPECTED_RAW_RECORDS:,} records but found "
        f"{len(records):,}. Check the miRBase release."
    )
if EXPECTED_SHA256 is not None:
    assert fasta_sha256 == EXPECTED_SHA256, (
        "The FASTA SHA-256 does not match the audited file. "
        f"Observed: {fasta_sha256}"
    )

print(f"Validated records: {len(records):,}")
print("SHA-256:", fasta_sha256)


## Global exact-sequence deduplication


In [ ]:
def globally_deduplicate_sequences(
    records: list[dict[str, str]],
) -> tuple[list[str], pd.DataFrame]:
    '''
    Return one copy of each exact RNA sequence plus an audit table.

    Sequences are sorted after deduplication so that all downstream files
    are deterministic and independent of Python set iteration order.
    '''
    sequence_counts = Counter(record["sequence"] for record in records)
    unique_sequences = sorted(sequence_counts)

    duplicate_audit = pd.DataFrame(
        {
            "sequence": list(sequence_counts),
            "number_of_fasta_records": list(sequence_counts.values()),
        }
    ).sort_values(
        ["number_of_fasta_records", "sequence"],
        ascending=[False, True],
        ignore_index=True,
    )
    return unique_sequences, duplicate_audit


unique_sequences, duplicate_audit = globally_deduplicate_sequences(records)

if EXPECTED_UNIQUE_SEQUENCES is not None:
    assert len(unique_sequences) == EXPECTED_UNIQUE_SEQUENCES, (
        f"Expected {EXPECTED_UNIQUE_SEQUENCES:,} globally unique sequences "
        f"but found {len(unique_sequences):,}."
    )

raw_lengths = pd.Series(
    [len(record["sequence"]) for record in records], name="length"
)
unique_lengths = pd.Series(
    [len(sequence) for sequence in unique_sequences], name="length"
)

print(f"Raw records: {len(records):,}")
print(f"Globally unique sequences: {len(unique_sequences):,}")
print(f"Duplicate records removed: {len(records) - len(unique_sequences):,}")
print(
    "Unique-sequence length range:",
    int(unique_lengths.min()),
    "to",
    int(unique_lengths.max()),
    "nt; median",
    float(unique_lengths.median()),
    "nt",
)


In [ ]:
length_distribution = (
    pd.concat(
        [
            raw_lengths.value_counts().rename("raw_records"),
            unique_lengths.value_counts().rename(
                "globally_unique_sequences"
            ),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .sort_index()
    .rename_axis("length_nt")
    .reset_index()
)

audit_metrics = pd.DataFrame(
    [
        ("fasta_path", str(FASTA_PATH.resolve())),
        ("fasta_sha256", fasta_sha256),
        ("raw_records", len(records)),
        ("globally_unique_sequences", len(unique_sequences)),
        (
            "duplicate_records_removed",
            len(records) - len(unique_sequences),
        ),
        ("minimum_unique_sequence_length", int(unique_lengths.min())),
        ("maximum_unique_sequence_length", int(unique_lengths.max())),
        ("median_unique_sequence_length", float(unique_lengths.median())),
        ("valid_alphabet", "ACGU"),
    ],
    columns=["metric", "value"],
)

audit_metrics.to_csv(
    OUTPUT_DIR / "universal_sequence_repertoire_audit.csv", index=False
)
length_distribution.to_csv(
    OUTPUT_DIR / "universal_sequence_length_distribution.csv", index=False
)
duplicate_audit.to_csv(
    OUTPUT_DIR / "universal_exact_sequence_duplicate_audit.csv",
    index=False,
)

display(audit_metrics)
display(length_distribution)


## Overlapping positional k-mer counting

For motif \(m\), k-mer size \(k\), and zero-based start position \(i\):

\[
f_{m,i}(\%) =
100 \times
\frac{C_{m,i}}
{N_i},
\qquad
N_i = \#\{s:\lvert s\rvert \ge i+k\}.
\]

Here \(C_{m,i}\) is the number of globally unique sequences containing
motif \(m\) beginning at position \(i\), and \(N_i\) is the number of
sequences long enough to contribute a complete k-mer at that position.
Consequently, every frequency row must sum to 100%.


In [ ]:
def ordered_kmers(k: int, base_order: tuple[str, ...] = BASE_ORDER) -> list[str]:
    return [
        "".join(characters)
        for characters in itertools.product(base_order, repeat=k)
    ]


def overlapping_kmers(sequence: str, k: int) -> list[str]:
    return [
        sequence[position : position + k]
        for position in range(len(sequence) - k + 1)
    ]


# Explicit toy checks make overlap and zero-based indexing auditable.
assert overlapping_kmers("ACGUA", 2) == ["AC", "CG", "GU", "UA"]
assert overlapping_kmers("ACGUA", 3) == ["ACG", "CGU", "GUA"]


def positional_kmer_tables(
    sequences: list[str],
    k: int,
    base_order: tuple[str, ...] = BASE_ORDER,
) -> dict[str, pd.DataFrame | pd.Series]:
    '''
    Count overlapping k-mers by zero-based start position.

    The denominator at position i is the number of unique sequences with
    length >= i + k. Thus every eligible sequence contributes exactly one
    k-mer to that positional row.
    '''
    motifs = ordered_kmers(k, base_order)
    motif_to_column = {motif: column for column, motif in enumerate(motifs)}
    maximum_start_position = max(len(sequence) - k for sequence in sequences)

    counts_array = np.zeros(
        (maximum_start_position + 1, len(motifs)), dtype=np.int64
    )
    eligible_array = np.zeros(maximum_start_position + 1, dtype=np.int64)

    for sequence in sequences:
        for position in range(len(sequence) - k + 1):
            motif = sequence[position : position + k]
            counts_array[position, motif_to_column[motif]] += 1
            eligible_array[position] += 1

    positions = pd.Index(
        range(maximum_start_position + 1), name="start_position_0based"
    )
    counts = pd.DataFrame(
        counts_array, index=positions, columns=motifs, dtype="int64"
    )
    eligible = pd.Series(
        eligible_array,
        index=positions,
        name="eligible_unique_sequences",
        dtype="int64",
    )
    frequency_percent = counts.div(eligible, axis=0).mul(100.0)

    # High-value invariants: one motif per eligible sequence and 100% per row.
    if not counts.sum(axis=1).equals(eligible):
        raise AssertionError(
            f"At least one positional count row for k={k} does not sum "
            "to its eligibility denominator."
        )
    if not np.allclose(
        frequency_percent.sum(axis=1).to_numpy(),
        np.full(len(frequency_percent), 100.0),
        atol=1e-10,
    ):
        raise AssertionError(
            f"At least one positional frequency row for k={k} "
            "does not sum to 100%."
        )

    return {
        "counts": counts,
        "eligible": eligible,
        "frequency_percent": frequency_percent,
    }


In [ ]:
results = {
    k: positional_kmer_tables(unique_sequences, k)
    for k in K_VALUES
}

for k, tables in results.items():
    counts = tables["counts"]
    frequencies = tables["frequency_percent"]
    eligible = tables["eligible"]
    print(
        f"k={k}: {len(counts.columns)} motifs; "
        f"positions {counts.index.min()}..{counts.index.max()}; "
        f"first-position denominator {int(eligible.iloc[0]):,}"
    )
    assert list(counts.columns) == ordered_kmers(k)
    assert not counts.isna().any().any()
    assert not frequencies.isna().any().any()


## Export complete count and frequency tables


In [ ]:
def export_kmer_tables(
    k: int,
    tables: dict[str, pd.DataFrame | pd.Series],
    output_dir: Path = OUTPUT_DIR,
) -> pd.DataFrame:
    counts = tables["counts"].copy()
    frequencies = tables["frequency_percent"].copy()
    eligible = tables["eligible"].copy()

    counts.to_csv(output_dir / f"universal_{k}mer_counts_wide.csv")
    frequencies.to_csv(
        output_dir / f"universal_{k}mer_frequency_percent_wide.csv",
        float_format="%.10f",
    )
    eligible.to_frame().to_csv(
        output_dir / f"universal_{k}mer_eligible_sequences.csv"
    )

    count_long = (
        counts.reset_index()
        .melt(
            id_vars="start_position_0based",
            var_name="kmer",
            value_name="count",
        )
    )
    frequency_long = (
        frequencies.reset_index()
        .melt(
            id_vars="start_position_0based",
            var_name="kmer",
            value_name="frequency_percent",
        )
    )
    tidy = count_long.merge(
        frequency_long,
        on=["start_position_0based", "kmer"],
        validate="one_to_one",
    )
    tidy = tidy.merge(
        eligible.rename_axis("start_position_0based")
        .reset_index(),
        on="start_position_0based",
        validate="many_to_one",
    )
    tidy.insert(
        1,
        "nucleotide_start_1based",
        tidy["start_position_0based"] + 1,
    )
    tidy.insert(
        2,
        "nucleotide_end_1based",
        tidy["start_position_0based"] + k,
    )
    tidy["k"] = k
    tidy[
        [
            "k",
            "start_position_0based",
            "nucleotide_start_1based",
            "nucleotide_end_1based",
            "kmer",
            "count",
            "eligible_unique_sequences",
            "frequency_percent",
        ]
    ].to_csv(
        output_dir / f"universal_{k}mer_positional_tidy.csv",
        index=False,
        float_format="%.10f",
    )
    return tidy


tidy_results = {
    k: export_kmer_tables(k, tables)
    for k, tables in results.items()
}

for k in K_VALUES:
    display(tidy_results[k].head())


## Editable SVG heatmaps


In [ ]:
# Preserve text as text in SVG so labels remain editable in Inkscape.
mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "font.family": FONT_FAMILY,
        "axes.labelcolor": "#252525",
        "xtick.color": "#252525",
        "ytick.color": "#252525",
        "text.color": "#252525",
    }
)
sns.set_theme(style="white", context="paper")


def supported_plot_positions(
    eligible: pd.Series,
    total_unique_sequences: int,
    minimum_fraction: float = MIN_ELIGIBLE_FRACTION_FOR_PLOT,
    maximum_override: int | None = MAX_START_POSITION_OVERRIDE,
) -> pd.Index:
    if maximum_override is not None:
        maximum = min(int(maximum_override), int(eligible.index.max()))
        return eligible.index[eligible.index <= maximum]

    minimum_n = int(np.ceil(total_unique_sequences * minimum_fraction))
    supported = eligible.index[eligible >= minimum_n]
    if len(supported) == 0:
        raise ValueError(
            "No position satisfies the minimum eligibility threshold."
        )
    return supported


def heatmap_size(number_of_motifs: int, number_of_positions: int) -> tuple[float, float]:
    # 2-mers remain portrait like the reference; 3-mers receive enough
    # horizontal room to keep all 64 labels separately editable.
    width = max(7.2, 0.275 * number_of_motifs + 3.0)
    height = max(6.0, 0.34 * number_of_positions + 2.0)
    return width, height


def plot_positional_frequency_heatmap(
    k: int,
    tables: dict[str, pd.DataFrame | pd.Series],
    output_dir: Path = OUTPUT_DIR,
) -> Path:
    frequencies = tables["frequency_percent"]
    eligible = tables["eligible"]
    positions = supported_plot_positions(
        eligible=eligible,
        total_unique_sequences=len(unique_sequences),
    )
    plot_data = frequencies.loc[positions]

    figure_width, figure_height = heatmap_size(
        number_of_motifs=plot_data.shape[1],
        number_of_positions=plot_data.shape[0],
    )
    fig, ax = plt.subplots(
        figsize=(figure_width, figure_height),
        constrained_layout=True,
    )
    fig.patch.set_facecolor(FIGURE_BACKGROUND)
    ax.set_facecolor(FIGURE_BACKGROUND)

    sns.heatmap(
        plot_data,
        ax=ax,
        cmap=COLOR_MAP,
        vmin=0,
        vmax=float(np.nanmax(plot_data.to_numpy())),
        linewidths=0.55,
        linecolor=GRID_COLOR,
        square=False,
        rasterized=False,
        cbar_kws={
            "label": "Frequency (%)",
            "shrink": 0.68,
            "pad": 0.025,
        },
    )

    ax.set_xlabel(f"{k}-mer", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel(
        "K-mer start position (0-based)",
        fontsize=13,
        fontweight="bold",
        labelpad=10,
    )
    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=90 if k == 3 else 0,
        ha="center",
        fontsize=7 if k == 3 else 10,
        fontweight="bold",
    )
    ax.set_yticklabels(
        [str(position) for position in positions],
        rotation=0,
        fontsize=9,
    )
    ax.tick_params(axis="both", length=0)

    colorbar = ax.collections[0].colorbar
    colorbar.ax.set_ylabel(
        "Frequency (%)",
        rotation=90,
        fontsize=12,
        fontweight="bold",
        labelpad=14,
    )
    colorbar.ax.tick_params(labelsize=9)

    output_path = output_dir / (
        f"universal_{k}mer_positional_frequency_RdPu.svg"
    )
    fig.savefig(
        output_path,
        format="svg",
        bbox_inches="tight",
        facecolor=FIGURE_BACKGROUND,
        transparent=SVG_TRANSPARENT,
        metadata={
            "Title": (
                f"Universal positional frequency of overlapping {k}-mers"
            ),
            "Description": (
                "Globally deduplicated miRBase mature-miRNA sequence "
                "repertoire; frequencies normalized by eligible sequences "
                "at each start position."
            ),
        },
    )
    plt.show()

    minimum_n = int(
        np.ceil(
            len(unique_sequences) * MIN_ELIGIBLE_FRACTION_FOR_PLOT
        )
    )
    print(
        f"Saved {output_path.resolve()} | displayed positions "
        f"{positions.min()}..{positions.max()} | minimum eligible n="
        f"{minimum_n:,}"
    )
    return output_path


In [ ]:
svg_paths = {
    k: plot_positional_frequency_heatmap(k, results[k])
    for k in K_VALUES
}
svg_paths


## Final reproducibility report


In [ ]:
provenance = {
    "analysis": "universal positional k-mer bias",
    "estimand": "globally unique mature-miRNA sequence repertoire",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_fasta": str(FASTA_PATH.resolve()),
    "input_sha256": fasta_sha256,
    "raw_records": len(records),
    "globally_unique_sequences": len(unique_sequences),
    "duplicate_records_removed": len(records) - len(unique_sequences),
    "valid_bases": "".join(BASE_ORDER),
    "k_values": list(K_VALUES),
    "overlapping_kmers": True,
    "frequency_denominator": (
        "number of globally unique sequences eligible to contribute a "
        "complete k-mer at each zero-based start position"
    ),
    "plot_minimum_eligible_fraction": (
        MIN_ELIGIBLE_FRACTION_FOR_PLOT
    ),
    "plot_maximum_start_position_override": MAX_START_POSITION_OVERRIDE,
    "motif_order": {
        str(k): ordered_kmers(k) for k in K_VALUES
    },
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": mpl.__version__,
        "seaborn": sns.__version__,
    },
    "svg_files": {
        str(k): str(path.resolve()) for k, path in svg_paths.items()
    },
}

provenance_path = OUTPUT_DIR / "universal_kmer_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

expected_files = [
    OUTPUT_DIR / "universal_sequence_repertoire_audit.csv",
    OUTPUT_DIR / "universal_sequence_length_distribution.csv",
    OUTPUT_DIR / "universal_exact_sequence_duplicate_audit.csv",
    provenance_path,
]
for k in K_VALUES:
    expected_files.extend(
        [
            OUTPUT_DIR / f"universal_{k}mer_counts_wide.csv",
            OUTPUT_DIR / f"universal_{k}mer_frequency_percent_wide.csv",
            OUTPUT_DIR / f"universal_{k}mer_eligible_sequences.csv",
            OUTPUT_DIR / f"universal_{k}mer_positional_tidy.csv",
            OUTPUT_DIR
            / f"universal_{k}mer_positional_frequency_RdPu.svg",
        ]
    )

missing_outputs = [path for path in expected_files if not path.is_file()]
if missing_outputs:
    raise AssertionError(f"Missing expected outputs: {missing_outputs}")

print("Analysis complete. Verified output files:")
for path in expected_files:
    print(" -", path.resolve())
